# Case Study 1: Word2Vec

Skip-gram with negative sampling, implemented from scratch, compared with gensim Word2Vec, gensim FastText, and **Polarity-Aware Word2Vec (PA-W2V)**: negation binding plus fold-nested polarity counter-fitting.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PACKAGES = ["gensim"]
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    if PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    hits = glob.glob("/kaggle/input/**/telelens/__init__.py", recursive=True)
    if hits:
        ROOT = Path(hits[0]).parents[2]
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BharatChandra-sys/NLP-case-study.git", "/tmp/repo"], check=True)
        ROOT = Path("/tmp/repo")
    RESULTS = Path("/kaggle/working/results/word2vec")
else:
    ROOT = Path.cwd().resolve().parent
    RESULTS = ROOT / "results" / "word2vec"
sys.path.insert(0, str(ROOT / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


def save_json(name, payload):
    (RESULTS / name).write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")


In [ ]:
from gensim.models import FastText, Word2Vec
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from telelens.data import load_simlex, load_telecom
from telelens.embeddings import Embeddings, log_odds_polarity, polarity_counterfit, train_sgns
from telelens.sentiment import dense_lr, grouped_folds
from telelens.text import bind_negation, tokenize
from telelens import viz

frame = load_telecom(10000)
plain = [tokenize(t) for t in frame["text"]]
bound = [bind_negation(d) for d in plain]
labels = frame["label"].to_numpy()
folds = grouped_folds(frame)
print(len(frame), "reviews", sum(map(len, plain)), "tokens", len({t for d in plain for t in d}), "types")
print("negation bound tokens", sum(t.startswith("not_") for d in bound for t in d))

In [ ]:
SYNONYMS = [("broadband", "internet"), ("engineer", "technician"), ("bill", "invoice"), ("rude", "unhelpful"),
            ("quick", "fast"), ("helpful", "friendly"), ("terrible", "awful"), ("cancel", "terminate"),
            ("cheap", "affordable"), ("router", "modem"), ("happy", "pleased"), ("problem", "issue"),
            ("refund", "reimbursement"), ("poor", "bad"), ("excellent", "great"), ("advisor", "agent")]
ANTONYMS = [("good", "bad"), ("fast", "slow"), ("helpful", "unhelpful"), ("reliable", "unreliable"),
            ("stable", "unstable"), ("cheap", "expensive"), ("polite", "rude"), ("happy", "unhappy"),
            ("excellent", "terrible"), ("easy", "difficult"), ("love", "hate"), ("best", "worst"),
            ("great", "poor"), ("friendly", "rude"), ("satisfied", "dissatisfied"), ("efficient", "useless")]
NEGATED = [("good", "not_good"), ("reliable", "not_reliable"), ("helpful", "not_helpful"), ("recommend", "not_recommend"),
           ("working", "not_working"), ("happy", "not_happy"), ("fast", "not_fast"), ("worth", "not_worth"),
           ("resolved", "not_resolved"), ("fixed", "not_fixed")]
HELD_OUT = frozenset(SYNONYMS + ANTONYMS + NEGATED)
simlex = load_simlex()


def pair_hits(emb, pairs, k=10):
    ranks = []
    for a, b in pairs:
        if a in emb and b in emb:
            ordered = [w for w, _ in emb.neighbours(a, len(emb.words))]
            ranks.append(ordered.index(b) + 1)
    if not ranks:
        return np.nan, np.nan
    return float(np.mean([r <= k for r in ranks])), float(np.mean([1 / r for r in ranks]))


def pair_metrics(emb):
    syn_hit, syn_mrr = pair_hits(emb, SYNONYMS)
    ant_hit, ant_mrr = pair_hits(emb, ANTONYMS)
    neg_hit, _ = pair_hits(emb, NEGATED)
    syn = [emb.cosine(a, b) for a, b in SYNONYMS]
    ant = [emb.cosine(a, b) for a, b in ANTONYMS]
    neg = [emb.cosine(a, b) for a, b in NEGATED]
    syn, ant, neg = [[x for x in values if x is not None] for values in (syn, ant, neg)]
    rows = [(s, 1) for s in syn] + [(a, 0) for a in ant]
    sims = [(emb.cosine(a, b), g) for a, b, g in simlex.itertuples(index=False)]
    sims = [(c, g) for c, g in sims if c is not None]
    return {"syn_hit10": syn_hit, "syn_mrr": syn_mrr, "ant_hit10": ant_hit, "ant_mrr": ant_mrr, "neg_hit10": neg_hit,
            "syn_mean": np.mean(syn), "ant_mean": np.mean(ant), "neg_mean": np.mean(neg) if neg else np.nan,
            "syn_ant_auc": roc_auc_score([r[1] for r in rows], [r[0] for r in rows]),
            "simlex_rho": spearmanr([s[0] for s in sims], [s[1] for s in sims]).statistic, "simlex_pairs": len(sims)}


def neighbour_purity(emb, held_out_polarity, z=1.96, k=10):
    polar = {w: s for w, s in held_out_polarity.items() if abs(s) >= z and w in emb}
    words = list(polar)
    ids = [emb.index[w] for w in words]
    sims = emb.vectors[ids] @ emb.vectors[ids].T
    np.fill_diagonal(sims, -1)
    signs = np.sign([polar[w] for w in words])
    top = np.argsort(-sims, axis=1)[:, :k]
    return float((signs[top] == signs[:, None]).mean())


def gensim_embeddings(model):
    return Embeddings(model.wv.index_to_key, model.wv.vectors.copy())


def train_all(idx):
    tr_plain = [plain[i] for i in idx]
    tr_bound = [bound[i] for i in idx]
    y = labels[idx]
    sgns, loss = train_sgns(tr_plain, device=DEVICE)
    bound_sgns, _ = train_sgns(tr_bound, device=DEVICE, bound=True)
    polarity = log_odds_polarity(tr_bound, y)
    pa, stats = polarity_counterfit(bound_sgns, polarity, excluded_pairs=HELD_OUT, device=DEVICE)
    settings = dict(vector_size=200, window=5, min_count=5, sg=1, negative=10, epochs=5, seed=SEED, workers=4)
    gw2v = gensim_embeddings(Word2Vec(tr_plain, **settings))
    fast = gensim_embeddings(FastText(tr_plain, min_n=3, max_n=6, **settings))
    return {"SGNS (scratch)": sgns, "gensim Word2Vec": gw2v, "FastText": fast, "SGNS + negation": bound_sgns, "PA-W2V": pa}, stats, loss

## Tune counter-fitting on an inner validation split of fold 0

In [ ]:
train_idx = folds[0][0]
inner_fit, inner_val = train_test_split(train_idx, test_size=0.2, stratify=labels[train_idx], random_state=SEED)
base, _ = train_sgns([bound[i] for i in inner_fit], device=DEVICE, bound=True)
inner_polarity = log_odds_polarity([bound[i] for i in inner_fit], labels[inner_fit])
grid = []
for repel_weight, attract_weight, anchor_weight in [(1.0, 0.3, 2.0), (1.0, 0.3, 1.0), (2.0, 0.3, 1.0), (1.0, 0.1, 4.0), (0.5, 0.3, 4.0), (2.0, 0.6, 2.0)]:
    emb, _ = polarity_counterfit(base, inner_polarity, excluded_pairs=HELD_OUT, device=DEVICE, repel_weight=repel_weight,
                                 attract_weight=attract_weight, anchor_weight=anchor_weight)
    x_fit, x_val = emb.embed_many([bound[i] for i in inner_fit]), emb.embed_many([bound[i] for i in inner_val])
    prob = dense_lr(x_fit, labels[inner_fit], x_val)
    grid.append({"repel": repel_weight, "attract": attract_weight, "anchor": anchor_weight,
                 "val_macro_f1": f1_score(labels[inner_val], prob >= 0.5, average="macro"), **pair_metrics(emb)})
x_fit, x_val = base.embed_many([bound[i] for i in inner_fit]), base.embed_many([bound[i] for i in inner_val])
print("no counter-fit val macro-F1", f1_score(labels[inner_val], dense_lr(x_fit, labels[inner_fit], x_val) >= 0.5, average="macro"))
tuning = pd.DataFrame(grid).sort_values("val_macro_f1", ascending=False)
tuning.to_csv(RESULTS / "counterfit_tuning.csv", index=False)
tuning.round(4)

## Five-fold evaluation with label use nested inside each training fold

In [ ]:
rows, stats_rows, losses = [], [], None
start = time.time()
for fold, (tr, te) in enumerate(folds):
    models, stats, loss = train_all(tr)
    losses = losses or loss
    stats_rows.append({"fold": fold, **stats})
    held_plain = log_odds_polarity([plain[i] for i in te], labels[te])
    held_bound = log_odds_polarity([bound[i] for i in te], labels[te])
    for name, emb in models.items():
        heldout = held_bound if emb.bound else held_plain
        rows.append({"fold": fold, "model": name, "vocab": len(emb.words), "neighbour_purity": neighbour_purity(emb, heldout), **pair_metrics(emb)})
    print("fold", fold, round(time.time() - start), "s")
fold_table = pd.DataFrame(rows)
fold_table.to_csv(RESULTS / "intrinsic_by_fold.csv", index=False)
summary = fold_table.drop(columns="fold").groupby("model", sort=False).agg(["mean", "std"]).round(4)
summary.to_csv(RESULTS / "intrinsic_summary.csv")
save_json("sgns_loss.json", losses)
pd.DataFrame(stats_rows).to_csv(RESULTS / "counterfit_constraints.csv", index=False)
summary

## Whole-corpus model: neighbours, negation probes and PCA

In [ ]:
all_idx = np.arange(len(frame))
final, final_stats, _ = train_all(all_idx)
QUERY = ["reliable", "unreliable", "not_reliable", "stable", "helpful", "engineer", "bill", "refund", "fast", "not_working"]
table = []
for word in QUERY:
    for name in ("SGNS + negation", "PA-W2V"):
        table.append({"word": word, "model": name, "neighbours": ", ".join(w for w, _ in final[name].neighbours(word, 6))})
neighbours = pd.DataFrame(table)
neighbours.to_csv(RESULTS / "neighbours.csv", index=False)

PROBES = [("the service is reliable", "the service is not reliable"), ("the connection is stable", "the connection is unstable"),
          ("the engineer was helpful", "the engineer was not helpful"), ("my problem was fixed", "my problem was not fixed"),
          ("broadband is slow and keeps disconnecting", "internet connection is sluggish and drops")]
probes = []
for a, b in PROBES:
    row = {"a": a, "b": b}
    for name, emb in final.items():
        tok = (lambda s: bind_negation(tokenize(s))) if emb.bound else tokenize
        row[name] = float(emb.embed(tok(a)) @ emb.embed(tok(b)))
    probes.append(row)
probes = pd.DataFrame(probes).round(3)
probes.to_csv(RESULTS / "doc_probes.csv", index=False)
display(neighbours.head(8), probes)

In [ ]:
WORDS = ["good", "great", "excellent", "helpful", "friendly", "reliable", "fast", "easy", "happy", "bad", "terrible", "awful", "rude",
         "unhelpful", "unreliable", "slow", "useless", "poor", "not_reliable", "not_helpful", "not_good", "not_working", "unstable", "stable"]
polarity = log_odds_polarity(bound, labels)
points = {}
for name in ("SGNS + negation", "PA-W2V"):
    emb = final[name]
    present = [w for w in WORDS if w in emb]
    xy = PCA(2, random_state=SEED).fit_transform(emb.vectors[[emb.index[w] for w in present]])
    points[name] = [(w, x, y, 1 if polarity.get(w, 0) > 0 else -1) for w, (x, y) in zip(present, xy)]
save_json("pca_points.json", points)
viz.pca_words(RESULTS / "word2vec_pca.png", points, "Selected word vectors, two PCA components")
means = fold_table.groupby("model", sort=False).mean(numeric_only=True)
viz.grouped(RESULTS / "syn_ant_cosine.png", list(means.index), [("Synonyms", means.syn_mean.tolist()), ("Antonyms", means.ant_mean.tolist())],
            "Mean cosine on held-out word pairs", "Cosine")
save_json("final_constraints.json", final_stats)

## Recorded Kaggle results

Executed on a Kaggle T4 GPU as `telelens-01-word2vec`. Files are in `results/word2vec/`.

**intrinsic_summary.csv**

|  | vocab | vocab.1 | neighbour_purity | neighbour_purity.1 | syn_mean | syn_mean.1 | ant_mean | ant_mean.1 | neg_mean | neg_mean.1 | syn_ant_auc | syn_ant_auc.1 | simlex_rho | simlex_rho.1 | simlex_pairs | simlex_pairs.1 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|  | mean | std | mean | std | mean | std | mean | std | mean | std | mean | std | mean | std | mean | std |
| model |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |
| SGNS (scratch) | 3757.6 | 24.2652 | 0.7738 | 0.0026 | 0.8325 | 0.0074 | 0.6949 | 0.0106 |  |  | 0.8167 | 0.0393 | 0.1069 | 0.0197 | 259.6 | 3.7815 |
| gensim Word2Vec | 3757.6 | 24.2652 | 0.7059 | 0.005 | 0.7279 | 0.0023 | 0.6455 | 0.0112 |  |  | 0.6908 | 0.0231 | 0.0665 | 0.0271 | 259.6 | 3.7815 |
| FastText | 3757.6 | 24.2652 | 0.7087 | 0.0091 | 0.7277 | 0.0049 | 0.732 | 0.0038 |  |  | 0.5392 | 0.0096 | 0.148 | 0.0107 | 259.6 | 3.7815 |
| SGNS + negation | 4049.8 | 28.2347 | 0.773 | 0.012 | 0.8279 | 0.0078 | 0.6827 | 0.0154 | 0.688 | 0.0302 | 0.8233 | 0.0463 | 0.0986 | 0.0116 | 251.2 | 2.49 |
| PA-W2V | 4049.8 | 28.2347 | 0.839 | 0.0212 | 0.7416 | 0.0154 | 0.3735 | 0.0174 | 0.5594 | 0.0608 | 0.8633 | 0.0203 | 0.0959 | 0.0189 | 251.2 | 2.49 |

**doc_probes.csv**

| a | b | SGNS (scratch) | gensim Word2Vec | FastText | SGNS + negation | PA-W2V |
|---|---|---|---|---|---|---|
| the service is reliable | the service is not reliable | 0.976 | 0.968 | 0.972 | 0.97 | 0.955 |
| the connection is stable | the connection is unstable | 0.99 | 0.982 | 0.994 | 0.987 | 0.957 |
| the engineer was helpful | the engineer was not helpful | 0.971 | 0.961 | 0.967 | 0.959 | 0.946 |
| my problem was fixed | my problem was not fixed | 0.975 | 0.961 | 0.966 | 0.968 | 0.963 |
| broadband is slow and keeps disconnecting | internet connection is sluggish and drops | 0.962 | 0.924 | 0.937 | 0.965 | 0.928 |

**counterfit_tuning.csv**

| repel | attract | anchor | val_macro_f1 | syn_mean | ant_mean | neg_mean | syn_ant_auc | simlex_rho | simlex_pairs |
|---|---|---|---|---|---|---|---|---|---|
| 1.0 | 0.3 | 1.0 | 0.888 | 0.7428 | 0.4892 | 0.5846 | 0.7125 | 0.091 | 229.0 |
| 2.0 | 0.3 | 1.0 | 0.8868 | 0.7377 | 0.4881 | 0.585 | 0.7083 | 0.0889 | 229.0 |
| 2.0 | 0.6 | 2.0 | 0.8867 | 0.7429 | 0.4878 | 0.5848 | 0.7125 | 0.091 | 229.0 |
| 0.5 | 0.3 | 4.0 | 0.8861 | 0.7856 | 0.5365 | 0.608 | 0.7333 | 0.0832 | 229.0 |
| 1.0 | 0.1 | 4.0 | 0.8848 | 0.7472 | 0.5011 | 0.5896 | 0.7083 | 0.0854 | 229.0 |
| 1.0 | 0.3 | 2.0 | 0.8836 | 0.745 | 0.4911 | 0.59 | 0.7083 | 0.0888 | 229.0 |

![syn_ant_cosine.png](../results/word2vec/syn_ant_cosine.png)

![word2vec_pca.png](../results/word2vec/word2vec_pca.png)

